# Plant Cell ABA Signaling Network ETL

- **Source**: `plantCellSignaling.data` (CFD Research Corp., based on Li et al. 2006)
- **Structure**: 43 nodes with Boolean states (0/1), 21 time steps per simulation (first row = random initial state), simulations separated by `*` lines
- **Pipeline**: Extract (parse & validate) → Transform (wide/tidy) → Load (CSV·Parquet·SQLite, re-verified)

In [1]:
from pathlib import Path
import re
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 50)

DATA_DIR = Path(r"C:/Users/USER/Desktop/[16-08-26] Cloudy/02.Abscisic Acid Signaling Network")
RAW_PATH = DATA_DIR / "plantCellSignaling.data"
OUT_DIR = DATA_DIR / "output"
OUT_DIR.mkdir(exist_ok=True)

RAW_PATH, OUT_DIR

(WindowsPath('C:/Users/USER/Desktop/[16-08-26] Cloudy/02.Abscisic Acid Signaling Network/plantCellSignaling.data'),
 WindowsPath('C:/Users/USER/Desktop/[16-08-26] Cloudy/02.Abscisic Acid Signaling Network/output'))

## 1. Extract — Parse and Validate the Raw File

The first line is a tab-separated list of node names; each block separated by `*` lines is one simulation (trajectory).

In [2]:
SEP = re.compile(r"^\*+$")


def extract_raw(raw_path: Path):
    """Parse the raw .data file: return (node name list, simulation blocks)."""
    lines = raw_path.read_text().splitlines()
    nodes = [n.strip() for n in lines[0].split("\t") if n.strip()]
    blocks: list[list[str]] = []
    current: list[str] = []
    for line in lines[1:]:
        if SEP.match(line):
            if current:
                blocks.append(current)
                current = []
        else:
            current.append(line)
    if current:
        blocks.append(current)
    return nodes, blocks


nodes, blocks = extract_raw(RAW_PATH)
print(f"Nodes: {len(nodes)}")
print(f"Simulations: {len(blocks)}  (note: .names document says 300; file actually contains 260)")
print(f"Rows per block: min={min(map(len, blocks))}, max={max(map(len, blocks))}")

Nodes: 43
Simulations: 260  (note: .names document says 300; file actually contains 260)
Rows per block: min=17, max=21


In [3]:
def validate(nodes, blocks):
    issues = []
    n = len(nodes)
    for sim_id, block in enumerate(blocks, start=1):
        for step, row in enumerate(block):
            if len(row) != n:
                issues.append((sim_id, step, f"length {len(row)} != {n}"))
            elif not re.fullmatch(r"[01]+", row):
                issues.append((sim_id, step, "non-binary characters"))
    return issues


issues = validate(nodes, blocks)
print(f"Structure issues: {len(issues)}")
issues[:10]

Structure issues: 0


[]

## 2. Transform — Build DataFrames

- **wide**: simulation x step x node (43 columns) — for trajectory analysis
- **long (tidy)**: `sim_id, step, node, state` — for SQL / visualization

In [4]:
def to_wide(nodes, blocks) -> pd.DataFrame:
    records = []
    for sim_id, block in enumerate(blocks, start=1):
        for step, row in enumerate(block):
            rec = {"sim_id": sim_id, "step": step}
            rec.update({node: int(ch == "1") for node, ch in zip(nodes, row)})
            records.append(rec)
    return pd.DataFrame(records)


wide = to_wide(nodes, blocks)
print(f"wide: {wide.shape[0]:,} rows x {wide.shape[1]} cols")
wide.head()

wide: 5,456 rows x 45 cols


,sim_id,step,ABA,CLOSURE,Ca,CaATPase,CAIM,CIS,ABH1,GCR,ERA1,PEPC,OST,SPHK,PH,PLD,ROP2,KEV,AGB,RAC,RCN,NIA12,PLC,InsPK,IP6,ADPRc,GC,NOS,S1P,PA,IP3,cADPR,cGMP,MALATE,ATRBOH,GPA,ROS,AnionEM,ABI,DEPOLAR,HATPase,KOUT,KAP,Actin,NO
0,1,0,1,0,1,1,1,0,1,1,1,0,1,0,1,1,0,1,1,0,1,1,0,1,0,0,1,0,1,0,0,0,1,0,1,1,0,0,0,0,1,1,0,0,1
1,1,1,1,1,0,0,0,0,1,1,1,0,1,1,1,1,0,1,1,0,1,1,0,1,1,1,1,1,1,1,0,1,1,0,0,1,0,1,1,1,0,1,0,1,1
2,1,2,1,1,0,0,0,1,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0,1,1,0,1,0,0,0,1,1,1,0,1,0,1,0,1,0
3,1,3,1,1,0,0,0,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0,1,1,0,0,0,0,1,1,1,1,0,1,0,1,0,1,0
4,1,4,1,1,0,0,0,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0,1,1,0,0,0,0,1,1,1,1,0,1,0,1,0,1,0


In [5]:
long = (
    wide.melt(id_vars=["sim_id", "step"], var_name="node", value_name="state")
    .sort_values(["sim_id", "step", "node"])
    .reset_index(drop=True)
)
print(f"long: {long.shape[0]:,} rows x {long.shape[1]} cols")
long.head(10)

long: 234,608 rows x 4 cols


,sim_id,step,node,state
0,1,0,ABA,1
1,1,0,ABH1,1
2,1,0,ABI,0
3,1,0,ADPRc,0
4,1,0,AGB,1
5,1,0,ATRBOH,1
6,1,0,Actin,0
7,1,0,AnionEM,0
8,1,0,CAIM,1
9,1,0,CIS,0


## 3. Quality Checks

- Detect constant nodes (.names document mentions 5 constant nodes)
- Check whether each simulation reaches a fixed point (last two steps identical)
- Enumerate unique steady states (attractors)

In [6]:
constant_nodes = [n for n in nodes if wide[n].nunique() == 1]
print(f"Constant nodes over the whole period: {len(constant_nodes)} -> {constant_nodes}")

last_step = wide.groupby("sim_id")["step"].transform("max")
steady = wide[wide["step"] == last_step].sort_values("sim_id").reset_index(drop=True)


def is_converged(blk: pd.DataFrame) -> bool:
    return (blk.iloc[-1, 2:].values == blk.iloc[-2, 2:].values).all()


conv_count = sum(
    is_converged(wide[wide.sim_id == sid].sort_values("step"))
    for sid in wide["sim_id"].unique()
)
print(f"Fixed point reached (last two steps identical): {conv_count}/{wide['sim_id'].nunique()}")

steady_patterns = steady.drop(columns=["sim_id", "step"]).drop_duplicates().reset_index(drop=True)
print(f"Unique steady states (attractors): {len(steady_patterns)}")

Constant nodes over the whole period: 2 -> ['ABA', 'AGB']


Fixed point reached (last two steps identical): 216/260
Unique steady states (attractors): 34


In [7]:
dist = (
    steady.groupby(nodes, sort=False)
    .agg(sim_count=("sim_id", "nunique"))
    .reset_index()
    .sort_values("sim_count", ascending=False)
)
print("Number of simulations converging to each attractor (top 10)")
dist.head(10)

Number of simulations converging to each attractor (top 10)


,ABA,CLOSURE,Ca,CaATPase,CAIM,CIS,ABH1,GCR,ERA1,PEPC,OST,SPHK,PH,PLD,ROP2,KEV,AGB,RAC,RCN,NIA12,PLC,InsPK,IP6,ADPRc,GC,NOS,S1P,PA,IP3,cADPR,cGMP,MALATE,ATRBOH,GPA,ROS,AnionEM,ABI,DEPOLAR,HATPase,KOUT,KAP,Actin,NO,sim_count
0,1,1,0,0,0,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0,1,1,0,0,0,0,1,1,1,1,0,1,0,1,0,1,0,223
2,1,1,0,0,0,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,1,1,0,1,1,0,1,1,0,1,1,1,1,0,1,0,1,0,1,0,3
5,1,1,1,0,0,1,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0,1,1,0,1,1,0,1,1,1,1,0,1,0,1,0,1,0,2
4,1,1,0,0,0,1,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0,1,1,0,0,0,0,1,1,1,1,0,1,0,1,0,1,0,2
1,1,1,0,0,0,1,1,1,1,0,1,1,1,1,1,1,1,0,1,1,0,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,0,1,0,1,0,1,1,1
3,1,1,0,0,0,1,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0,1,1,1,0,0,0,1,1,1,1,0,1,0,1,0,1,0,1
6,1,1,0,1,0,1,1,1,1,0,1,1,1,1,1,1,1,0,1,1,0,1,1,1,1,1,1,1,1,0,1,0,1,1,1,1,0,1,0,1,0,1,1,1
7,1,1,0,1,0,1,1,1,1,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,1,0,0,0,1,1,1,1,0,1,0,1,0,1,1,1
8,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,0,1,0,1,1,1,1,1,0,0,1,1,1,1,0,0,0,1,1,1,1,0,1,0,1,0,1,0,1
9,1,1,1,0,0,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,0,1,1,1,1,1,1,1,0,0,0,0,1,1,1,1,0,1,0,1,0,1,1,1


## 4. Load — Store and Re-verify

- CSV: wide / long
- Parquet: wide / long
- SQLite: `trajectory_wide`, `trajectory_long`, `steady_states`

In [8]:
wide.to_csv(OUT_DIR / "plant_cell_signaling_wide.csv", index=False)
long.to_csv(OUT_DIR / "plant_cell_signaling_long.csv", index=False)
wide.to_parquet(OUT_DIR / "plant_cell_signaling_wide.parquet", index=False)
long.to_parquet(OUT_DIR / "plant_cell_signaling_long.parquet", index=False)

db_path = OUT_DIR / "plant_cell_signaling.db"
with sqlite3.connect(db_path) as con:
    wide.to_sql("trajectory_wide", con, if_exists="replace", index=False)
    long.to_sql("trajectory_long", con, if_exists="replace", index=False)
    steady.to_sql("steady_states", con, if_exists="replace", index=False)

for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name:40s} {f.stat().st_size:>10,} bytes")

.ipynb_checkpoints                                0 bytes
plant_cell_signaling.db                   4,411,392 bytes
plant_cell_signaling_long.csv             3,293,452 bytes
plant_cell_signaling_long.parquet            29,804 bytes
plant_cell_signaling_wide.csv               508,221 bytes
plant_cell_signaling_wide.parquet            45,121 bytes


In [9]:
parquet_back = pd.read_parquet(OUT_DIR / "plant_cell_signaling_wide.parquet")
pd.testing.assert_frame_equal(wide, parquet_back)

with sqlite3.connect(db_path) as con:
    sqlite_back = pd.read_sql(
        "SELECT * FROM trajectory_wide ORDER BY sim_id, step", con
    ).reset_index(drop=True)
pd.testing.assert_frame_equal(wide.reset_index(drop=True), sqlite_back, check_dtype=False)

print("Re-load verification passed: Parquet / SQLite both match the original DataFrame")

Re-load verification passed: Parquet / SQLite both match the original DataFrame


Authors: Jerry Jenkins, Abhishek Soni
